In [23]:
import squidpy as sq
import cellcharter as cc
import scanpy as sc
import pandas as pd
import numpy as np

flag = True
df_val = pd.DataFrame()


### Shape Characterization

In [24]:

def cellcharter_metrics(data_name, category, data_path, ctp_path):

    global flag, df_val
    
    adata = sc.read_h5ad(data_path)    
    ctp = pd.read_csv(ctp_path)
    print ("ctp shape", ctp.shape, "and", ctp.columns)

    if (adata.shape[0] != ctp.shape[0]):
        adata = adata[adata.obs_names.isin(ctp[ctp.columns[0]])]
        
    print ("adata shape", adata.shape)
    
    for c in ctp.columns[1:]:
        threshold = np.median(ctp[c].values)
        adata.obs[c + "_spatial_cluster"] = (ctp[c] > threshold).astype(int).values
    
    sq.gr.spatial_neighbors(adata, coord_type = 'generic', delaunay = True) # library_key='sample',
    
    for c in ctp.columns[1:]:
        
        cc.gr.connected_components(adata, cluster_key = c + '_spatial_cluster')
        cc.tl.boundaries(adata, min_hole_area_ratio = 0.1)
        cc.tl.linearity(adata)
        cc.tl.curl(adata)
        cc.tl.elongation(adata)
    
        adata.obs.rename(columns = {'component': c +'_component'}, inplace = True)
        #print (adata)
        mapping = pd.DataFrame({"Act" : adata.obs[c + "_spatial_cluster"].values, "Pred" : adata.obs[c + "_component"].values})
        mapping = mapping.drop_duplicates()
    
        #adata.uns[c + '_shape_component'] = adata.uns.pop('shape_component')
    
        t = mapping.drop_duplicates()
        t = t[t["Act"] == 1]
        t.dropna(inplace = True)
        
        comp_itr = 1
        for comp in t["Pred"].values:
            
            linearity = adata.uns["shape_component"]["linearity"][comp]
            curl = adata.uns["shape_component"]["curl"][comp]
            elongation = adata.uns["shape_component"]["elongation"][comp]
    
            if (flag): 
                df_val["Category"] = [category]
                df_val["Dataset"] = [data_name]
                df_val["cell_type"] = [c + "_" + str(comp_itr)]
                df_val["linearity"] = [linearity]
                df_val["curl"] = [curl]
                df_val["elongation"] = [elongation]
                
                #print (df_val)
                comp_itr += 1
                flag = False
            else:            
                new_row = {"Category" : category, "Dataset" : data_name, "cell_type": c + "_" + str(comp_itr), "linearity" : linearity, "curl" : curl, "elongation" : elongation}
                comp_itr += 1
                df_val = df_val.append(new_row, ignore_index = True)
                #print (df_val) 
                
    #print (df_val)


### Brain

In [5]:
lst = {
        "Brain" : ["151674", "151673", "151508", "151509", "151669", "151670"],
      "Cancer": [],
      "Development" : []
        }


for l in lst["Brain"]:
    #data_path = "/data/Ajita/Spatial/Datasets/Spatial_Deconvolution/" + l + "/ST.h5ad"
    data_path = "/data/Ajita/Spatial/Datasets/Spatial_Deconvolution/DLPFC/simulator_outputs/simulated_gene_expression" + l + "_average.h5ad"
    ctp_path = "/data/Ajita/Spatial/Datasets/Spatial_Deconvolution/DLPFC/simulator_outputs/Ground_Truth_CTP_" + l + ".csv"
    
    cellcharter_metrics(l, "Brain", data_path, ctp_path)

In [6]:
print (df_val)

    Category Dataset           cell_type  linearity      curl  elongation
0      Brain  151674            AST_FB_1   0.661855  0.000000    0.141729
1      Brain  151674            AST_PP_1   0.928212  0.370839    0.118858
2      Brain  151674       Endothelial_1   0.639627  0.000000    0.141729
3      Brain  151674             IN_PV_1   0.694891  0.057353    0.147707
4      Brain  151674            IN_SST_1   0.697082  0.066228    0.054699
..       ...     ...                 ...        ...       ...         ...
116    Brain  151670        Neu_NRGN_I_1   0.736368  0.116647    0.207436
117    Brain  151670       Neu_NRGN_II_1   1.000000  0.000000    0.358445
118    Brain  151670           Neu_mat_1   0.657105  0.000000    0.401130
119    Brain  151670               OPC_1   1.000000  0.344929    0.114159
120    Brain  151670  Oligodendrocytes_1   0.791708  0.198444    0.072249

[121 rows x 6 columns]
